# Chest X-Ray Pneumonia — GitHub + Colab (no Drive for code)

Repo: [bachnguyennn/Chest_X-Pneumonia-Detection](https://github.com/bachnguyennn/Chest_X-Pneumonia-Detection)

1. **Runtime → GPU** → Run all cells top to bottom
2. Paste Kaggle API key in the dataset cell (no file upload)

> Research/education only — not for clinical use.

In [1]:
import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch 2.10.0+cu128 | CUDA: True
GPU: Tesla T4


## 1. Clone from GitHub

In [2]:
# Your GitHub repo (already configured)
GITHUB_USER = "bachnguyennn"
REPO_NAME = "Chest_X-Pneumonia-Detection"
REPO_URL = "https://github.com/bachnguyennn/Chest_X-Pneumonia-Detection.git"

import os
import sys
from pathlib import Path
PROJECT_ROOT = Path("/content") / REPO_NAME

if PROJECT_ROOT.exists() and (PROJECT_ROOT / "src" / "train.py").exists():
    print(f"Repo already at {PROJECT_ROOT}")
else:
    if PROJECT_ROOT.exists():
        !rm -rf {PROJECT_ROOT}
    !git clone {REPO_URL} {PROJECT_ROOT}

assert (PROJECT_ROOT / "src" / "train.py").exists(), f"Clone failed: {REPO_URL}"

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "chest_xray"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR = PROJECT_ROOT / "models"
BATCH_SIZE = 16

for d in (FIGURES_DIR, MODELS_DIR, DATA_ROOT.parent):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src/ OK:", True)

Cloning into '/content/Chest_X-Pneumonia-Detection'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 32 (delta 5), reused 32 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 37.15 KiB | 3.10 MiB/s, done.
Resolving deltas: 100% (5/5), done.
PROJECT_ROOT: /content/Chest_X-Pneumonia-Detection
src/ OK: True


## 2. Install dependencies

In [3]:
# Colab-safe install (avoids jupyter version conflicts)
%pip install -q torch torchvision timm grad-cam scikit-learn matplotlib seaborn tqdm Pillow kaggle opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 93.1 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 3. Download dataset (Kaggle)

Accept rules: [Chest X-Ray Pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
Set `KAGGLE_USERNAME` and `KAGGLE_KEY` in the next cell (from [Kaggle Settings → API](https://www.kaggle.com/settings)). **Do not commit your key to GitHub.**

In [4]:
import json
import os

# ============ PASTE YOUR KAGGLE API (session only — never push to GitHub) ============
KAGGLE_USERNAME = "bachnguyennn"   # your Kaggle username
KAGGLE_KEY = "KGAT_a9cb0f07ec09d35cb1d6564c45975630"                    # paste key here, e.g. KGAT_...
# ===================================================================================

if (DATA_ROOT / "train").exists():
    print(f"Dataset already at {DATA_ROOT}")
else:
    if not KAGGLE_KEY:
        raise ValueError("Paste your KAGGLE_KEY in this cell, then re-run.")
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "w") as f:
        json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("Kaggle credentials saved.")
    print("Downloading ~1.2 GB from Kaggle (10–30 min)...")
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {DATA_ROOT.parent} --unzip
    print("Done.")

from src.dataset import get_dataset_stats
import json as _json
print(_json.dumps(get_dataset_stats(DATA_ROOT), indent=2))

Kaggle credentials saved.
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:58<00:00, 42.4MB/s]

Done.
{
  "train": {
    "NORMAL": 1341,
    "PNEUMONIA": 3875
  },
  "val": {
    "NORMAL": 8,
    "PNEUMONIA": 8
  },
  "test": {
    "NORMAL": 234,
    "PNEUMONIA": 390
  }
}


## 4. Train (~20–40 min on T4)

In [5]:
from src.train import train

history = train(
    data_root=DATA_ROOT,
    backbone="resnet50",
    epochs=15,
    batch_size=BATCH_SIZE,
    lr=1e-4,
    output_dir=MODELS_DIR,
    num_workers=2,
)

Device: cuda
Dataset stats: {
  "train": {
    "NORMAL": 1341,
    "PNEUMONIA": 3875
  },
  "val": {
    "NORMAL": 8,
    "PNEUMONIA": 8
  },
  "test": {
    "NORMAL": 234,
    "PNEUMONIA": 390
  }
}
Class weights: [1.4858129024505615, 0.5141870975494385]
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 185MB/s]


Parameters: 4,098 trainable / 23,512,130 total


Epoch 1/15 (107.0s) | Train F1: 0.2261 | Val F1: 0.2222 | Val Recall: 0.1250
  -> Saved best model (F1=0.2222)


Epoch 2/15 (104.2s) | Train F1: 0.6864 | Val F1: 0.6667 | Val Recall: 0.5000
  -> Saved best model (F1=0.6667)


...

Epoch 11/15 (110.7s) | Train F1: 0.9654 | Val F1: 0.9412 | Val Recall: 1.0000


Epoch 12/15 (110.1s) | Train F1: 0.9665 | Val F1: 1.0000 | Val Recall: 1.0000
  -> Saved best model (F1=1.0000)


Epoch 13/15 (110.4s) | Train F1: 0.9668 | Val F1: 0.9412 | Val Recall: 1.0000


Epoch 14/15 (109.3s) | Train F1: 0.9698 | Val F1: 1.0000 | Val Recall: 1.0000


Epoch 15/15 (110.2s) | Train F1: 0.9733 | Val F1: 1.0000 | Val Recall: 1.0000

Training complete. Best Val F1: 1.0000
Model saved to: /content/Chest_X-Pneumonia-Detection/models/best_resnet50.pth


## 5. Evaluate + heatmaps

In [7]:
from src.evaluate import run_evaluation
from IPython.display import Image, display

CHECKPOINT = MODELS_DIR / "best_resnet50.pth"
results = run_evaluation(
    checkpoint_path=CHECKPOINT,
    data_root=DATA_ROOT,
    output_dir=FIGURES_DIR,
    split="test",
    generate_cams=True,
    num_cam_samples=8,
    num_failure_cases=4,
)
print(f"PR-AUC: {results['pr_auc']:.4f} | ROC-AUC: {results['roc_auc']:.4f}")

Evaluating: 100%|██████████| 20/20 [00:11<00:00,  1.77it/s]


              precision    recall  f1-score   support

      NORMAL       0.93      0.83      0.88       234
   PNEUMONIA       0.90      0.96      0.93       390

    accuracy                           0.91       624
   macro avg       0.92      0.90      0.91       624
weighted avg       0.91      0.91      0.91       624

PR AUC: 0.9777
ROC AUC: 0.9697


Generating CAMs:   0%|          | 0/20 [00:01<?, ?it/s]


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
for name in ["confusion_matrix.png", "precision_recall_curve.png"]:
    p = FIGURES_DIR / name
    if p.exists():
        display(Image(filename=str(p), width=500))

cam_dir = FIGURES_DIR / "cam_comparisons"
if cam_dir.exists():
    for p in sorted(cam_dir.glob("*.png"))[:3]:
        display(Image(filename=str(p), width=700))

## 6. Save results (before session ends)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/pneumonia_results", "zip", PROJECT_ROOT / "models")
shutil.make_archive("/content/pneumonia_figures", "zip", FIGURES_DIR)
files.download("/content/pneumonia_results.zip")
files.download("/content/pneumonia_figures.zip")
print("Downloaded model + figures to your computer.")